# Reto 01 — Hito 1: Auditoría inicial de la gestión de residuos municipales (INEC, 2024)

**Asignatura:** Aprendizaje Automático Inicial · **Carrera:** Ciencia de Datos · **Integrantes:** completar.

**Pregunta:** ¿Qué perfiles de gestión municipal pueden identificarse posteriormente y cómo comunicar sus diferencias?

**Alcance actual:** auditoría, elección justificada de cinco características y plan de preparación. **Aquí no se entrena K-Means ni se calcula PCA.**

**Evidencia:** este notebook **requiere el CSV real**. Hasta ejecutarlo, las cifras específicas del informe recibido de otro análisis deben considerarse pendientes de verificación. Se conserva el archivo original sin alterarlo.

## 1. Obtener el CSV oficial de 2024

Fuente: [Portal GAD Municipales — INEC](https://www.ecuadorencifras.gob.ec/gad-municipales/). Seleccionar 2024 → Gestión Integral de Residuos Sólidos → Bases de datos (CSV), diccionario, formulario y guía de uso.

El enlace oficial identificado para el ZIP es:
`https://www.ecuadorencifras.gob.ec/documentos/web-inec/Encuestas_Ambientales/Municipios/2024/grs/2_BDD_DATOS_ABIERTOS_GRS_GADM_2024_12_CSV.zip`

La celda intentará descargarlo. Si falla, en Colab pedirá subir manualmente el ZIP o el archivo `BDD_GIRS_2024_VF.csv`. **No sustituir por 2023/2025.**

In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import io, zipfile, hashlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

URL = 'https://www.ecuadorencifras.gob.ec/documentos/web-inec/Encuestas_Ambientales/Municipios/2024/grs/2_BDD_DATOS_ABIERTOS_GRS_GADM_2024_12_CSV.zip'
carpeta = Path('/content/girs2024') if Path('/content').exists() else Path.cwd() / 'girs2024'
carpeta.mkdir(exist_ok=True)
fecha_descarga = None
origen = None

try:
    import requests
    respuesta = requests.get(URL, timeout=60, headers={'User-Agent':'Mozilla/5.0'})
    respuesta.raise_for_status()
    if not zipfile.is_zipfile(io.BytesIO(respuesta.content)):
        raise ValueError('El enlace no devolvió un ZIP válido')
    zip_path = carpeta / 'datos_originales_2024.zip'
    zip_path.write_bytes(respuesta.content)
    origen = str(URL)
    fecha_descarga = datetime.now(ZoneInfo('America/Guayaquil')).isoformat(timespec='seconds')
    print('Descarga oficial completada:', zip_path)
except Exception as exc:
    print('No se descargó automáticamente:', type(exc).__name__, str(exc)[:200])
    print('Sube el ZIP/CSV de 2024 desde el portal oficial.')
    try:
        from google.colab import files
        subidos = files.upload()
        if not subidos: raise FileNotFoundError('No se subió archivo')
        nombre, contenido = next(iter(subidos.items()))
        zip_path = carpeta / nombre
        zip_path.write_bytes(contenido)
        origen = 'Archivo 2024 subido manualmente desde el portal INEC'
        fecha_descarga = 'Fecha de obtención manual: completar por el equipo'
    except ImportError:
        raise RuntimeError('Ejecuta esta celda en Colab o copia el CSV/ZIP a la carpeta girs2024.')

In [ ]:
if zipfile.is_zipfile(zip_path):
    with zipfile.ZipFile(zip_path) as z:
        opciones = [n for n in z.namelist() if n.lower().endswith('.csv')]
        print('Archivos CSV del ZIP:', opciones)
        candidatas = [n for n in opciones if 'GIRS_2024' in n.upper() or 'GRS_2024' in n.upper()]
        if not candidatas:
            raise FileNotFoundError('No aparece un CSV correspondiente a GIRS/GRS 2024 en el ZIP.')
        ruta_archivo = carpeta / Path(candidatas[0]).name
        ruta_archivo.write_bytes(z.read(candidatas[0]))
else:
    ruta_archivo = zip_path

if '2024' not in ruta_archivo.name:
    raise ValueError('El nombre del archivo no permite confirmar la edición 2024; revisar manualmente.')
print('Archivo:', ruta_archivo.name)
print('Origen:', origen, '| Fecha:', fecha_descarga)
print('SHA-256:', hashlib.sha256(ruta_archivo.read_bytes()).hexdigest())

## 2. Lectura y trazabilidad

Parámetros exigidos por el profesor: punto y coma como separador, coma decimal, Windows-1252 y `CantonId` como texto. La existencia de 221 × 213 es una **referencia a verificar**, no una orden para recortar datos.

In [ ]:
df = pd.read_csv(ruta_archivo, sep=';', decimal=',', encoding='cp1252', dtype={'CantonId':'string'}, low_memory=False)
df.columns = df.columns.astype(str).str.replace('\ufeff', '', regex=False).str.strip()
print('Dimensiones observadas:', df.shape)
assert 'CantonId' in df, 'El archivo no contiene CantonId; comprobar su formato/edición.'
if df.shape != (221, 213):
    print('ATENCIÓN: las dimensiones difieren de la referencia del Reto 01; revisar origen y carga.')
print('Nombres de columnas duplicados:', df.columns[df.columns.duplicated()].tolist())
display(df.head(3))

## 3. Auditoría estructural

Diferenciar claves repetidas, filas totalmente idénticas y nombres de cantón homónimos. El nombre geográfico no es identificador único: cantones de provincias diferentes pueden llamarse igual.

In [ ]:
resumen = pd.DataFrame([
    ['Filas',len(df)],['Columnas',len(df.columns)],
    ['CantonId nulos',int(df['CantonId'].isna().sum())],
    ['CantonId distintos',int(df['CantonId'].nunique(dropna=True))],
    ['CantonId duplicados',int(df['CantonId'].duplicated().sum())],
    ['Filas íntegramente duplicadas',int(df.duplicated().sum())],
    ['Celdas faltantes',int(df.isna().sum().sum())],
    ['% celdas faltantes',round(100*df.isna().sum().sum()/df.size,2)]
],columns=['Comprobación','Resultado'])
display(resumen)
if 'Provin' in df:
    print('Provincias distintas en Provin:',df['Provin'].nunique(dropna=True))
col_nombre = next((c for c in ['NombreCantón','NombreCanton','NOMBRE_CANTON'] if c in df),None)
if col_nombre:
    print('Nombres de cantón distintos:',df[col_nombre].nunique(dropna=True))
    display(df.loc[df[col_nombre].duplicated(keep=False),['CantonId','Provin',col_nombre] if 'Provin' in df else ['CantonId',col_nombre]].sort_values(col_nombre))

In [ ]:
calidad = pd.DataFrame({
    'columna':df.columns,
    'tipo_importado':df.dtypes.astype(str).to_numpy(),
    'n_faltantes':df.isna().sum().to_numpy(),
    'porcentaje_cobertura':(100*df.notna().mean()).round(1).to_numpy(),
    'valores_distintos':df.nunique(dropna=True).to_numpy()
})
display(calidad.sort_values(['porcentaje_cobertura','columna']).head(25))
print('Columnas ≥90% de cobertura:',int((calidad.porcentaje_cobertura>=90).sum()))
print('Columnas <50% de cobertura:',int((calidad.porcentaje_cobertura<50).sum()))
print('Columnas sin variación en los datos presentes:')
display(calidad.loc[calidad.valores_distintos<=1,['columna','n_faltantes','valores_distintos']])

## 4. Diagnóstico cuantitativo de los campos candidatos

*No convertir códigos nominales (`MR441`, `MR442`, `MR461`) en cantidades*. Para las magnitudes continuas, inspeccionar faltantes, ceros, negativos y máximos. Estas cifras solo son correctas si la base se ha importado desde la edición requerida.

In [ ]:
continuas=['VB','MR4223','CSB','CRR','TROT','RI','RIR','MR4321','PORG','DIF','MR453','NODIF','MR4341_TOTAL_RES_REC']
categoricas=['MR441','MR442','MR452','MR4521','MR461']
def numerica(nombre):
    if nombre not in df: return pd.Series(np.nan,index=df.index,dtype='float64')
    s=df[nombre]
    if pd.api.types.is_numeric_dtype(s): return pd.to_numeric(s,errors='coerce')
    return pd.to_numeric(s.astype('string').str.strip().str.replace(',','.',regex=False),errors='coerce')
num={c:numerica(c) for c in continuas+categoricas}
filas=[]
for c in continuas:
    s=num[c]
    filas.append({'Variable':c,'Existe':c in df,'Válidos':int(s.notna().sum()),
                  'Vacíos':int(s.isna().sum()),'Ceros':int(s.eq(0).sum()),
                  'Negativos':int(s.lt(0).sum()),'Mínimo':s.min(),'Máximo':s.max()})
diagnostico=pd.DataFrame(filas)
display(diagnostico)

In [ ]:
fig,ax=plt.subplots(1,3,figsize=(14,3.5))
for a,c in zip(ax,['CSB','CRR','RI']):
    s=num[c].dropna()
    a.hist(s,bins=25)
    a.set_title(c+' (datos originales)');a.set_ylabel('Municipios')
plt.tight_layout();plt.show()

## 5. Casos que no deben corregirse automáticamente

Comparar `CSB > 1`, `CSB` vacío con denominador y vías barridas; `DIF/MR453 > 1`; zeros de `RI`; y consistencia de `TROT` con otras magnitudes. El informe comparado menciona municipios específicos: **confirmarlos únicamente si aparecen en la salida**.

In [ ]:
contexto=[c for c in ['CantonId','NombreCantón','NombreCanton','Provin'] if c in df]
casos_csb = num['CSB'].isna() | (num['CSB']>1) | ((num['MR4223']<=0)&num['MR4223'].notna())
print('CSB nulos:',int(num['CSB'].isna().sum()),'CSB > 1:',int((num['CSB']>1).sum()),
      'MR4223 ≤ 0:',int((num['MR4223']<=0).sum()))
display(pd.concat([df.loc[casos_csb,contexto], pd.DataFrame({c:num[c][casos_csb] for c in ['VB','MR4223','CSB']})],axis=1))
# Un campo vacío debido a un salto del formulario no se reemplaza por 0 automáticamente.
if 'MR452' in df and 'MR4521' in df:
    saltos=(num['MR452']==2)&df['MR4521'].isna()
    print('MR452=No y MR4521 vacío:',int(saltos.sum()))

In [ ]:
ratio_dif=(num['DIF']/num['MR453'].where(num['MR453']>0))
ratio_dif=ratio_dif.where(num['DIF']>=0)
ratio_ri=num['RI']/(1000*num['CRR'].where(num['CRR']>0))
ratio_ri=ratio_ri.where(num['RI']>=0)
ratio_organicos=num['TROT']/num['CRR'].where(num['CRR']>0)
controles=pd.DataFrame({
    'diferenciada_sobre_total':ratio_dif,
    'inorganicos_sobre_total':ratio_ri,
    'organicos_sobre_total':ratio_organicos,
    'RI_cero':num['RI'].eq(0),
    'DIF_mayor_total':ratio_dif>1,
    'TROT_mayor_recoleccion':ratio_organicos>1
})
print('DIF/MR453 > 1:',int(controles.DIF_mayor_total.sum()))
print('RI=0:',int(controles.RI_cero.sum()))
print('TROT > CRR:',int(controles.TROT_mayor_recoleccion.sum()))
casos=controles.DIF_mayor_total|controles.TROT_mayor_recoleccion
cols=[c for c in ['DIF','MR453','TROT','CRR','RI'] if c in df]
display(pd.concat([df.loc[casos,contexto+cols],controles.loc[casos,['diferenciada_sobre_total','organicos_sobre_total']]],axis=1))

## 6. Candidatas para la matriz futura (Hito 2)

Se preparan **solo indicadores provisionales**, para estudiar su cobertura y detectar anomalías:

1. `CSB`: proporción de barrido (revisar vacíos y valores > 1).
2. `DIF/MR453`: proporción diaria de recolección diferenciada (marcar > 1).
3. `MR441/MR442`: 1 si al menos una de las dos respuestas es sí; 0 solo si ambas son no.
4. `RI/(1000*CRR)`: intensidad inorgánica mensual (los ceros pueden requerir comprobación manual).
5. `MR461=3`: indicador de botadero principal (no asignar distancias a códigos 1/2/3).

Los casos contradictorios de recuperación **no pueden identificarse de forma automática únicamente a partir de `RI=0`**: cotejar el formulario y las respuestas por material antes de excluirlos.

In [ ]:
M=pd.DataFrame(index=df.index)
M['cobertura_barrido']=num['CSB']
M['porcion_diferenciada']=ratio_dif.where(ratio_dif.between(0,1,inclusive='both'))
a,b_=num['MR441'],num['MR442']
M['actividad_organica']=np.where(a.eq(1)|b_.eq(1),1,np.where(a.eq(2)&b_.eq(2),0,np.nan))
M['intensidad_inorganica']=ratio_ri.where(ratio_ri.between(0,1,inclusive='both'))
c=num['MR461']
M['botadero_principal']=np.where(c.eq(3),1,np.where(c.isin([1,2]),0,np.nan))
print('Cobertura de las cinco características candidatas (SIN imputación):')
display(M.notna().sum().to_frame('válidos'))
print('Matriz de casos completos antes de resolver ceros ambiguos:',int(M.notna().all(axis=1).sum()))
print('¡No afirmar todavía que la muestra es de 204 ni de 215 sin auditar los 11 ceros contradictorios!')

In [ ]:
# Inspeccionar correlación sin usarla todavía como criterio automático de eliminación
print('Spearman de dos indicadores binarios/candidatos:')
print(M[['porcion_diferenciada','actividad_organica']].corr(method='spearman'))

# Descriptores externos: no ingresarán por defecto al cálculo de distancias
externos=pd.DataFrame(index=df.index)
externos['CantonId']=df['CantonId'].astype('string')
for col in ['NombreCantón','NombreCanton','Provin']:
    if col in df: externos[col]=df[col]
externos['CRR_ton_mes']=num['CRR']
externos['PPC_urbana']=num['MR4321']
display(externos.head())

## 7. Exportación y plan de preparación

**Plan (todavía sin ejecutar clustering):**

- Inspeccionar ceros de `RI` frente a las preguntas de recuperación por material del formulario. Documentar qué casos tienen cero real, información contradictoria o ausencia.
- Mantener una matriz conservadora excluyendo solo los casos con indicadores indefinidos y los ceros contradictorios **identificados por revisión**, y una matriz de sensibilidad que conserve ceros defendibles.
- Revisar escalado de proporciones y variables binarias: estandarizar automáticamente variables binarias raras puede crear distancias artificialmente grandes.
- Analizar efecto de escala por separado con `log1p(CRR)`; no mezclar cantidad absoluta y componentes del mismo total sin justificación.
- Hito 2: comparar K-Means y jerárquico (mínimo 3 valores de k entre 2 y 6) y preparar PCA. La silueta se calcula en la matriz original de agrupamiento, no en los dos componentes PCA.

La salida de auditoría se guarda como CSV independientes; **ninguna celda sobrescribe el archivo original**.

In [ ]:
calidad.to_csv(carpeta/'auditoria_columnas_2024.csv',index=False,encoding='utf-8-sig')
diagnostico.to_csv(carpeta/'auditoria_continuas_2024.csv',index=False,encoding='utf-8-sig')
M.to_csv(carpeta/'caracteristicas_provisionales_2024.csv',index=False,encoding='utf-8-sig')
externos.to_csv(carpeta/'descriptores_2024.csv',index=False,encoding='utf-8-sig')
resumen.to_csv(carpeta/'resumen_auditoria_2024.csv',index=False,encoding='utf-8-sig')
print('Exportaciones disponibles en:',carpeta)

## 8. Conclusión del Hito 1

La base es útil para proponer perfiles exploratorios, pero la elección de las variables exige separar cantidades absolutas, proporciones, categorías y valores ausentes. Se recomienda priorizar prácticas de gestión y reservar la escala municipal para interpretar o contrastar los grupos. Los números de municipios de una futura matriz **deben salir de este notebook ejecutado**, no copiarse de una propuesta sin verificar.

**Fuentes oficiales:** [INEC GAD Municipales](https://www.ecuadorencifras.gob.ec/gad-municipales/) · [ANDA GIRS 2024](https://anda.inec.gob.ec/anda5/index.php/catalog/1294) · [Guía de uso](https://www.ecuadorencifras.gob.ec/documentos/web-inec/Encuestas_Ambientales/Municipios/2024/grs/Guia_de_Uso_de_la_BDD_GIRS.pdf).